<div dir="rtl" style="text-align:right">
<h1>Attention را خانه‌به‌خانه باز کنیم</h1><p style="text-align:right"><b>پرسش آزمایش:</b> وزن هر منبع از کجا می‌آید و چه چیزی را جابه‌جا می‌کند؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-05/chapter-01/28-qkv.html"><bdi dir="ltr">28-qkv</bdi></a>، <a href="http://127.0.0.1:8000/part-05/chapter-02/29-scores.html"><bdi dir="ltr">29-scores</bdi></a>، <a href="http://127.0.0.1:8000/part-05/chapter-02/30-scaling.html"><bdi dir="ltr">30-scaling</bdi></a>، <a href="http://127.0.0.1:8000/part-05/chapter-02/31-values.html"><bdi dir="ltr">31-values</bdi></a>، <a href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html"><bdi dir="ltr">33-mask</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، Kernel را Restart و سپس Run All کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این یک نمونهٔ آموزشی دستی است، نه وزن‌های آموزش‌دیدهٔ پروژه. Tokenها فقط برچسب موقعیت‌اند. همهٔ مرحله‌ها را آشکار می‌سازیم؛ کتابخانهٔ Attention آماده‌ای فراخوانی نمی‌شود. قبل از اجرا حدس بزنید سطر آخر کدام Key را سازگارتر می‌بیند.</p>
</div>

In [ ]:
import math
tokens = ["a","b","c"]
embeddings = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])  # (T,C)
W_Q = torch.eye(2)
W_K = torch.tensor([[1.,1.],[2.,0.]])
W_V = torch.tensor([[1.,0.],[0.,2.]])
def attention_math(x):
    q, k, v = x @ W_Q.T, x @ W_K.T, x @ W_V.T
    raw = q @ k.T
    scaled = raw / math.sqrt(q.shape[-1])
    allowed = torch.ones(x.shape[0],x.shape[0],dtype=torch.bool).tril()
    masked = scaled.masked_fill(~allowed, float("-inf"))
    weights = torch.softmax(masked, dim=-1)
    output = weights @ v
    return dict(embeddings=x, Q=q, K=k, V=v, raw_scores=raw,
                scaled_scores=scaled, mask=allowed, masked_scores=masked,
                weights=weights, output=output)
trace = attention_math(embeddings)
print("Tokens:", tokens)
for name, value in trace.items():
    inspect(name, value)
    print(value)
torch.testing.assert_close(trace["weights"].sum(-1), torch.ones(3))
assert torch.count_nonzero(trace["weights"].triu(1)) == 0
manual_last = sum(trace["weights"][-1,j] * trace["V"][j] for j in range(3))
torch.testing.assert_close(manual_last, trace["output"][-1])


<div dir="rtl" style="text-align:right">
<h2>از جدول عدد به نقشهٔ وزن</h2><p style="text-align:right">هر سطر یک Query و هر ستون یک Key است. رنگ، ضریب ترکیب Value را نشان می‌دهد؛ به‌تنهایی علت کامل پاسخ مدل یا رابطهٔ دستوریِ تضمین‌شده نیست.</p>
</div>

In [ ]:
def heatmap(weights, title, ax):
    plot = ax.imshow(weights.detach().numpy(), vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=range(3), yticks=range(3), xticklabels=tokens, yticklabels=tokens,
           xlabel="Key", ylabel="Query", title=title)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f"{weights[i,j].item():.2f}",ha="center",va="center",
                    color="white" if weights[i,j] > 0.5 else "black")
    return plot

changed = embeddings.clone()
changed[1] = torch.tensor([2.,1.])  # One controlled input change.
new_trace = attention_math(changed)
fig, axes = plt.subplots(1,2,figsize=(8,3))
heatmap(trace["weights"], "Original", axes[0])
heatmap(new_trace["weights"], "Changed token b", axes[1])
plt.tight_layout()
plt.show()
print("Output change:", new_trace["output"]-trace["output"])
torch.testing.assert_close(new_trace["output"][0],trace["output"][0])


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> فقط W_V را دو برابر کنید و تابع را دوباره اجرا کنید. انتظار داریم وزن‌ها ثابت بمانند و خروجی دو برابر شود؛ چرا؟ سپس تغییر در X را با تغییر در W_V مقایسه کنید: در اولی Q و K هم می‌توانند عوض شوند. علت ثابت‌ماندن سطر اول در تغییر Token دوم را در دفتر بعد جدا می‌آزماییم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>